In [1]:
import numpy as np

def calc_cg_intermediate(freq, depth, g=9.81, niter=4):
    """
    Calculate linear-wave group velocity c_g for intermediate water.

    Parameters
    ----------
    freq : float or array-like
        Frequency (Hz)
    depth : float or array-like
        Water depth (m), positive
    g : float
        Gravitational acceleration (m/s^2)
    niter : int
        Number of Newton iterations

    Returns
    -------
    cg : float or ndarray
        Group velocity (m/s)

        Shapes:
        - scalar if freq and depth are scalar
        - (nf,) if freq is 1D and depth scalar
        - (np,) if freq scalar and depth is 1D
        - (np, nf) if both are 1D
    """
    f = np.atleast_1d(np.asarray(freq, dtype=float))
    h = np.atleast_1d(np.asarray(depth, dtype=float))

    omega = 2.0 * np.pi * f            # (nf,)
    om = omega[None, :]                # (1, nf)
    hh = h[:, None]                    # (np, 1)

    # deep-water initial guess
    k = np.maximum(om**2 / g, 1e-10)

    for _ in range(niter):
        kh = k * hh
        tanh_kh = np.tanh(kh)
        sech2_kh = 1.0 / np.cosh(kh)**2
        F = g * k * tanh_kh - om**2
        dFdk = g * tanh_kh + g * k * hh * sech2_kh
        k = k - F / dFdk

    c = om / k
    n = 0.5 * (1.0 + (2.0 * k * hh) / np.sinh(2.0 * k * hh))
    cg = n * c

    # return shape that matches the input style
    freq_scalar = np.isscalar(freq)
    depth_scalar = np.isscalar(depth)

    if freq_scalar and depth_scalar:
        return float(cg[0, 0])
    elif depth_scalar:
        return cg[0, :]     # (nf,)
    elif freq_scalar:
        return cg[:, 0]     # (np,)
    else:
        return cg           # (np, nf)
cg = calc_cg_intermediate( 1/25, [30, 25, 20])
print(cg)

[15.56099415 14.43972083 13.12791803]


In [2]:
# lowest frequency in some wave spectra
1/.038

26.315789473684212

In [3]:
# how much change with many iterations
cg = calc_cg_intermediate( 1/25, [30, 25, 20], niter=20 )
print(cg)

[15.56099474 14.43972442 13.12794174]


In [4]:
# Check deepwater approximation
cg = calc_cg_intermediate( .1, 1000. )
print(cg, 9.81/(4*np.pi*0.1) )

7.806549958657468 7.806549958657467
